# Tarea 1 - Métodos determinísticos para la física

### Emil Winkler Partida - A01352501
### 27/02/2026

## Método Newton - Rhampson



## Pregunta 2

 - ¿Que tipo de datos o clase tiene que ser cada uno de las entradas?
    - **function**: *Callable*. Acepta n argumentos numéricos escalares y devuelve un array
    - **variables**: *Matrix*: Matriz de variables rescatadas por el parcing del imput del usuario. 
    - **jac_function**: *Callable*. Acepta n argumentos numéricos escalares y devuelve una matriz jacobiana n x n
    - **initial_value**: *Array*. Lista de números float con valores iniciales del problema.
    - **tolerance**: *Float*. Valor de referencia numérico para definir el error.
    - **max_iterations**: *Integer*. Número entero de la cantidad de iteraciones máximas.
 - En los casos en los que las entradas son funciones, ¿Que definición tienen estas funciones?
    - Primero se definen a nivel simbólico con *sympy* para que puedan operarse las derivadas exactas del jacobiano. No obstante, una vez hecho este proceso matemático se transforman a funciones numéricas que son computacionalmente más rápidas(*sp.lambdify*).
    - *function* es un vector columna con las ecuaciones a resolver, por otro lado, *jac_function* representa la matriz de derivadas de estas mismas funciones.
 - ¿Cual será tu condición de salida del ciclo iterativo?
    - Se utiliza el residuo, que es la norma del vector de la función (magnitud del vector de error) como marco de referencia. Si el ultimo residuo calculado es menor al valor definido en *tolerance*, el proceso iterativo se detiene con un break y significa que se ha encontrado la raiz. 


## Pregunta 3
### Función de Jacobiano y de Newton-Rhapson


In [ ]:
# Importamos librerias importantes para el desarrollo del proyecto

import numpy as np
import sympy as sp
import math  
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor

# Función del Método Newton-Raphson para resolver sistemas de ecuaciones no lineales

def newton_raphson(function, variables, initial_value, tolerance = 1e-10, max_iterations = 1000):
    residuals = [] # Inicializamos una lista para almacenar residuos

    # Función para sacar Jacobiano para funciones de n dimensiones con n cantidad de variables
    jacobian_matriz = []
    if len(function) != len(variables):
        raise ValueError("El número de funciones debe ser igual al número de variables")
    for f in function:
        # Se saca la derivada de cada función con respecto  a cada variable con un for y ses agrega a la matriz del jacobiano
        jacobian_matriz.append([sp.diff(f,variables[i]) for i in range(len(variables))])  

    # Convertimos expresiones simbólicas a funciones numéricas con lamdbify para poder evaluarlas
    funct = sp.lambdify(variables, function, 'numpy') # Especificamos que queremos utilizar numpy para operaciones
    jac_function = sp.lambdify(variables, jacobian_matriz, 'numpy')

    # Creamos un arreglo que almacene los valores de las variables
    x_k = np.array(initial_value, dtype=float) 

    for _ in range(max_iterations):
        f_k = np.array(funct(*x_k), dtype=float).flatten() # Utilizamos flattern para convertir la funcion en un arreglo unidimensional para operar
        J_k = np.array(jac_function(*x_k), dtype=float) # Calculamos el jacobiano de la función en el punto 

        # Resolvemos el sistema de ecuaciones lineales de J_k * delta = -f_k
        try:
            delta = np.linalg.solve(J_k, -f_k) 
        except np.linalg.LinAlgError: # Si el jacobiano es singular, no se puede resolver el sistema
            print("El Jacobiano es singular, no es posible continuar con el método.")
            return None, residuals
        
        # Actualizamos el valor de la variable en base al hiperplano tangente
        x_k = x_k + delta 

        # Calculamos el residuo como norma de la función en el punto actual y lo guardamos en la lista
        residuals.append(np.linalg.norm(f_k)) 

        # Si el ultimo residuo es menor a la tolerancia, se retorna la solución y los residuos
        if residuals[-1] < tolerance:
            break
    
    # Evaluamos la funcion en el punto final
    final_evaluation = np.array(funct(*x_k), dtype = float).flatten()

    for i, result in enumerate(final_evaluation):
        print(f'Ecuacion {i+1} valuada en la solución: {result}')
        if abs(result) < tolerance: 
            print(f"Ya que el resultado se acerca a cero, se puede concluir que la solucion encontrada es correcta para la ecuación {i+1}. \n")

    return x_k, residuals


## Pregunta 4
###  Encuentra el punto de intersección de las curvas $y = cos(x),   y = x^3 -1$

Para comparar el resultado, evaluamos la función esperando que se acerque al 0. Si todas las ecuaciones evaluadas llegan a un 0 (o suficientemente cerca), se concluye que se encontro la ecuación del sistema de ecuaciones. 

In [ ]:
# Definimos variables arbitrarias para definir funciones a través de sympy
transformations = (standard_transformations + (implicit_multiplication_application, convert_xor))

# Añadimos un filtro para casos comunes de error con el usuario
filter = {'e': sp.E, 'exp': sp.exp, "pi": sp.pi, 'sqrt':sp.sqrt,}
input("Presina Enter para resolver el sistema de ecuaciones no lineales")

# Inicializamos lista para almacenar funciones
function = []
while True:
    try:
        f = input("Ingrese la función (en términos de x, y, z, etc.) o q para finalizar: ")
        if f.lower() == 'q':
            break
        function.append(parse_expr(f, transformations=transformations, local_dict = filter))
        f_filtered = f.replace(",", '')
    except Exception as e:
        print(f"Error enla expresión: {e}")

# Unimos todas las funciones que ingreso el usuario 
func_expresion = sp.Matrix(function)

print("Funciones ingresadas:", func_expresion)
# Extraemos variables de las funciones y las ordenamos alfabéticamente
variables = sorted(list(func_expresion.atoms(sp.Symbol)), key=lambda s: s.name )

print("Variables detectadas:", variables)
# if input("Estan correctas las variables? (s/n):"). lower() != 's':
initial_value = []
print("Ingrese los valores iniciales para: ")
for var in variables:
    initial_value.append(float(input(f"Valor inicial para {var}:")))

# Las ingresamos a una matriz
variables_matrix = sp.Matrix(variables)

# Solucionamos las ecuaciones con newton-raphson
solution, residuals = newton_raphson(function, variables_matrix, initial_value )

# Imprimimos la solución y el último residuo solo si este existe
if solution is not None and residuals is not None:
    print("Solución:", solution, "\nResiduos:", residuals[-1])
else:
    print("No se encontró una solución a el sistema de ecuaciones")
    

Funciones ingresadas: Matrix([[-y + cos(x)], [x**3 - y - 1]])
Variables detectadas: [x, y]
Ingrese los valores iniciales para: 
Ecuacion 1 valuada en la solución: 5.551115123125783e-17
Ya que el resultado se acerca a cero, se puede concluir que la solucion encontrada es correcta para la ecuación 1. 

Ecuacion 2 valuada en la solución: -4.440892098500626e-16
Ya que el resultado se acerca a cero, se puede concluir que la solucion encontrada es correcta para la ecuación 2. 

Solución: [1.12656191 0.42976673] 
Residuos: 6.753223014464259e-16


## Pregunta 5
### Encuentra el valor de ${x, y, z}$ que cumple con 
### $$z = 1 - 5e^-(\frac{x^2+y^2}{20})$$
### $$z = -x^2 - y^2$$
### $$ x + y = 1$$

Para comparar el resultado, evaluamos la función esperando que se acerque al 0. Si todas las ecuaciones evaluadas llegan a un 0 (o suficientemente cerca), se concluye que se encontro la ecuación del sistema de ecuaciones. 

In [ ]:
# Definimos variables arbitrarias para definir funciones a través de sympy
transformations = (standard_transformations + (implicit_multiplication_application, convert_xor))

# Añadimos un filtro para casos comunes de error con el usuario
filter = {'e': sp.E, 'exp': sp.exp, "pi": sp.pi, 'sqrt':sp.sqrt,}
input("Presina Enter para resolver el sistema de ecuaciones no lineales")

# Inicializamos lista para almacenar funciones (Estas funciones no deben de contener igual y deben estar igualadas a 0.)
function = []
while True:
    try:
        f = input("Ingrese la función (en términos de x, y, z, etc.) o q para finalizar: ")
        if f.lower() == 'q':
            break
        function.append(parse_expr(f, transformations=transformations, local_dict = filter))
        f_filtered = f.replace(",", '')
    except Exception as e:
        print(f"Error enla expresión: {e}")

# Unimos todas las funciones que ingreso el usuario 
func_expresion = sp.Matrix(function)

print("Funciones ingresadas:", func_expresion)
# Extraemos variables de las funciones y las ordenamos alfabéticamente
variables = sorted(list(func_expresion.atoms(sp.Symbol)), key=lambda s: s.name )

print("Variables detectadas:", variables)
# if input("Estan correctas las variables? (s/n):"). lower() != 's':
initial_value = []
print("Ingrese los valores iniciales para: ")
for var in variables:
    initial_value.append(float(input(f"Valor inicial para {var}:")))

# Las ingresamos a una matriz
variables_matrix = sp.Matrix(variables)

# Solucionamos las ecuaciones con newton-raphson
solution, residuals = newton_raphson(function, variables_matrix, initial_value )

# Imprimimos la solución y el último residuo solo si este existe
if solution is not None and residuals is not None:
    print("Solución:", solution, "\nResiduos:", residuals[-1])
else:
    print("No se encontró una solución a el sistema de ecuaciones")
    

Funciones ingresadas: Matrix([[-z - 5*exp(-x**2/20 - y**2/20) + 1], [-x**2 - y**2 - z], [x + y - 1]])
Variables detectadas: [x, y, z]
Ingrese los valores iniciales para: 
Ecuacion 1 valuada en la solución: 4.440892098500626e-16
Ya que el resultado se acerca a cero, se puede concluir que la solucion encontrada es correcta para la ecuación 1. 

Ecuacion 2 valuada en la solución: -4.440892098500626e-16
Ya que el resultado se acerca a cero, se puede concluir que la solucion encontrada es correcta para la ecuación 2. 

Ecuacion 3 valuada en la solución: 0.0
Ya que el resultado se acerca a cero, se puede concluir que la solucion encontrada es correcta para la ecuación 3. 

Solución: [-0.67261809  1.67261809 -3.25006635] 
Residuos: 5.068262347813025e-14


## Método Simpson


## Pregunta 6


- ¿Que tipo de datos o clase tiene que ser cada uno de las entradas?
    - **function**: *Callable*. Función matemática a integrar definidamente
    - **domain**: *List*. Contiene los elemenos *a* (float/int) =  límite inferior; *b* (float/int) = límite superior; *n* (int) = cantidad inicial de subintervalos.
    - **tolerance**: *Float*. Margen de error permitido 
- En los casos en los que las entradas son funciones, ¿Que definición tienen estas funciones?
    - En este caso, *function* representa una función a la cual se le aplicará la integral definida. Matemáticamente esta función debe ser capaz de recibir un argumento numérico y retornar un único valor numérico. Para esta aplicación se evaluara mucho esta función en base a iteraciones y el paso. 
- ¿Cuál será tu condición de salida del ciclo iterativo?
    - Este algoritmo calcula la integral con *n* intervalos y también la que tiene *2n* de subintervalos y las compara. Si el valor absoluto de la diferencia de estos resultados es menor a la *tolerance*, entonces se asume que el valor se estabilizo en una respuesta concreta y se cierra el ciclo while con un break.

## Pregunta 7
### Función para Método Simpson

In [28]:

def integrate(function, domain, tolerance = 1e-6):
    a, b, n = domain # Separamos el dominio en sus componentes

    # Condición del método para la cantidad de subintervalos 
    if n % 2 != 0:
        print("n debe ser un número par para el método de Simpson")
        return
    
    while True:
        
        # Definimos "pasos" o subintervalos a través del dominio
        h = (b-a)/n

        # Definimos la cantidad de subintervalos para la comparativa de tolerancia a traves de 2n
        h2 = (b-a)/(2*n)

        # Asignamos valores de coeficientes
        coefficients = [1 if k == 0 or k == (n) else 2 if k % 2 == 0 else 4 for k in range(n+1)] # Usamos n+1 para incluir el punto final
        coefficients2 = [1 if k == 0 or k == (2*n) else 2 if k % 2 == 0 else 4 for k in range(2*n+1)] 
        
        # Defimos la suma/ponderación que compone al método
        total_sum = sum(coefficients[i]*function(a + i*h) for i in range(n+1))
        total_sum2 = sum(coefficients2[i]*function(a + i*h2) for i in  range(2*n+1))

        # Agregamos el factor de multiplicación a la suma de mi integral con n subintervalos y con 2n subintervalos
        integral1, integral2 = (h/3)*total_sum, (h2/3)*total_sum2

        # Hacemos comparativa |integral2 - integral1| < tolerancia para determinar la precisión
        if abs(integral2 - integral1) < tolerance:
            print(f"El valor de la integral: {round(integral2,6)}, Subintervalos necesitados: {2*n}")
            break
        else:
            n *= 2

    return integral2    

## Pregunta 8
### Área de un círculo unitario
### $$x^2 + y^2 = 1$$ 

### Para estos ejemplos definimos *domain* como [a = limite inferior, b = limite superior, n = cantidad de subintervalos]

Además compararemos este resultado con el área calculada a través de
$$ A = \pi r^2 = \pi * 1^2 = \pi \approx 3.141592$$


In [29]:
# Función a integrar despejada para tener solo dependencia de x
function = lambda x: math.sqrt(1 - x**2) 

# Definimos el dominio del circulo unitario, con su n cantidad de subintervalos (Aunque estos puedan cambiar en base a la tolerancia)
# domain = [a = limite inferior, b = limite superior, n = cantidad de subintervalos]
domain = [-1, 1, 4]

# Calculamos la integral con el método de Simpson e imprimimos resultados (Notar que por la raiz solo tendremos la mitad positiva)
result = 2*integrate(function, domain, tolerance = 1e-6) # Multiplicamos por 2 para obtener el círculo completo
print(f"Resultado: {round(result, 6)}")

El valor de la integral: 1.570796, Subintervalos necesitados: 16384
Resultado: 3.141592


## Pregunta 9
### $$ \int_{-1}^{1} cosh(x)dx$$

Para comparar la respuesta, voy a comparar el método numérico con el resultado obtenido en Mathematica

### $$ \int_{-1}^{1} cosh(x)dx \approx 2.3504$$

In [30]:
# Función a integrar despejada para tener solo dependencia de x
function = lambda x: math.cosh(x) 

# Definimos el dominio del circulo unitario, con su n cantidad de subintervalos (Aunque estos puedan cambiar en base a la tolerancia)
# domain = [a = limite inferior, b = limite superior, n = cantidad de subintervalos]
domain = [-1, 1, 4]

# Calculamos la integral con el método de Simpson e imprimimos resultados (Notar que por la raiz solo tendremos la mitad positiva)
result = integrate(function, domain, tolerance = 1e-6) # Multiplicamos por 2 para obtener el círculo completo
print(f"Resultado: {round(result,6)}")

El valor de la integral: 2.350402, Subintervalos necesitados: 64
Resultado: 2.350402


## Pregunta 10 
### $$\int_{-4\pi}^{4\pi} \frac{sin(x)}{x} dx$$

Para este ejercicio definimos una condición en la función debido a la naturaleza de la misma. Como en este caso el dominio incluye el 0 donde se indetermina la función, se recurren a otros métodos para poder evaluar esta función. 

En este caso evaluaremos el límite con L'Hopital $f(x) = \frac{\sin(x)}{x}$ cuando $x \to 0$. 

$$
\begin{aligned}
\lim_{x \to 0} \frac{\sin(x)}{x} &= \lim_{x \to 0} \frac{\frac{d}{dx} (\sin x)}{\frac{d}{dx} (x)} \\
&= \lim_{x \to 0} \frac{\cos(x)}{1} \\
&= \cos(0) \\
&= 1
\end{aligned}
$$

Entonces cuando la funcion tiende a 0 
$$\lim_{x \to 0} \frac{\sin(x)}{x} = 1$$

Para comparar la respuesta, voy a comparar el método numérico con el resultado obtenido en Mathematica
### $$\int_{-4\pi}^{4\pi} \frac{sin(x)}{x} dx \approx 2.98432$$

In [31]:
# Función a integrar despejada para tener solo dependencia de x
function = lambda x: math.sin(x)/x if x != 0 else 1 # Evitamos indeterminación para esta función

# Definimos el dominio del circulo unitario, con su n cantidad de subintervalos (Aunque estos puedan cambiar en base a la tolerancia)
domain = [-4*math.pi, 4*math.pi, 4]

# Calculamos la integral con el método de Simpson e imprimimos resultados (Notar que por la raiz solo tendremos la mitad positiva)
result = integrate(function, domain, tolerance = 1e-6) # Multiplicamos por 2 para obtener el círculo completo
print(f"Resultado: {round(result,6)}")

El valor de la integral: 2.984322, Subintervalos necesitados: 512
Resultado: 2.984322


## Método Monte Carlo

## Pregunta 11

- ¿Que entradas debe tener la función? ¿Que tipo de datos o clase tiene que ser cada uno de las entradas? 
    - **num_dim**: *Integer*. Define el número de ejes o dimensiones del espacio de la hiperesfera.
    - **radius**: *Float*. Define la extensión de la hiperesfera desde el centro.
    - **limits**: *List*. Lista de tuplas con números reales (a,b) que define los límites para la región donde se tendran los puntos aleatorios.
    - **num_samples**: *Int*. Cantidad total de iteraciones. 
- ¿Cómo puedes generar una lista de números aleatorios? ¿Que tamaño tiene que tener esta lista?
    - Para hacer esto utilice un list comprehension para sacar puntos aleatorios con probabilidad uniforme entre el mínimo y el máximo de esa dimension. Cada vez que el ciclo en la lista de una vuelta se genera un solo punto. Básicamente la lista que genera un punto tiene una cantidad de n números para una n cantidad de dimensiones.
- ¿Cómo se define una hiperesfera? 
    - Como hablamos de un concepto generalizado y visualizado en 2D y 3D pero a N dimensiones, podemos definirlo como el conjunto de puntos en un espacio de n dimensiones que se encuentran a una distancia menor o igual a un radio *r* desde un punto central. 
    $$ x_1^2 + x_2^2 + . . . + x_n^2 \leq r^2$$
- ¿Que función tienes que integrar para encontrar su volumen?
    - Se integra una *función indicadora/característica* la cual sirve para definir si el dardo cayo dentro o fuera de la hiperesfera. Al promediar estos valores de 1 y 0 se multiplican por un volumen base y se calcula asi el volumen aproximado de la hiperesfera. 

    $$ f(x_1,...,x_n) = 1 \; si \;  x_1^2 + ... + x_n^2 \leq r^2$$
    $$ f(x_1,...,x_n) = 0 \;  si \;  x_1^2 + ... + x_n^2 > r^2$$

    Finalmente se calcula el volumen como la integral de esta función en el dominio del hipercubo


## Pregunta 12
### Función para Método Monte Carlo

In [40]:

def sphere_volume():

    try:
        # Pedimos al usuario ingrear la cantidad de dimensiones
        num_dim = int(input("Ingrese la cantidad en números enteros de dimensiones (N): "))
        radius = float(input("Ingrese el radio de la hiperesfera: "))
        num_samples = int(input("Ingrese el número de muestras (el. 1,000,000): "))

        # Limits es una lista de tuplas (min, max) para cada variable
        limits = [(-radius, radius) for _ in range(num_dim)] # Generamos limites de integración para cada variable

        # Podemos saber las dimensiones del espacio con la longitud de la lista
        dimensions = len(limits)
        print(f"Dimensiones del espacio: {dimensions}")
        
        # Función que evalua si un punto está dentro de la hiperesfera
        def hyper_radius(*coordinates):
            return 1 if sum(x**2 for x in coordinates) <= radius**2 else 0
        
        # Inicializamos el volumen del espacio del hiper-volumen
        volumen = 1 
        for a, b in limits:
            if a >= b:
                print("Los limites deben ser válidos (min < max)") # a debe ser menor que b para que el espacio tenga volumen 
                return None
            volumen *= (b-a) # Calculamos el volumen del espacio en base a los limites de integración 
        
        sum_dim = 0 # Inicializamos la suma donde se evaluará la función en los puntos aleatorios generados
        for _ in range(num_samples):
            # Generamos un punto aleatorio dentro de los limites de integración para cada dimensión
            rand_point = [np.random.uniform(a, b) for a, b in limits]
            # Evaluamos la función en ese punto para sumarla a un total
            sum_dim += hyper_radius(*rand_point)
        total = (sum_dim/num_samples) * volumen
        print(f'El volumen de la hiperesfera de {dimensions} dimensiones con radio {radius} es aproximadamente: {round(total, 6)}')
        # Retornamos el resultado de la integral, que se define como el promedio de las evaluaciones por el volumen del espacio
        return total
    
    except ValueError:
        print('Error: Por favor ingrese un número válido para las dimensiones y el número de muestras')


## Pregunta 13
### Área de un círculo 

Para definir el área de un circulo, definimos una hiperesfera de dos dimensiones. 


In [ ]:

# Llamamos a la función general que permite calcular cualquier hiperesfera
sphere_volume()

Dimensiones del espacio: 2
El volumen de la hiperesfera de 2 dimensiones con radio 1.0 es aproximadamente: 3.141616


3.141616

## Pregunta 14
### Volumen de una hiperesfera

Para calcular el volumen de una esfera, las dimensiones deben ser minimo 3.

In [ ]:

# Llamamos a la función general que permite calcular cualquier hiperesfera
sphere_volume()

Dimensiones del espacio: 4
El volumen de la hiperesfera de 4 dimensiones con radio 1.0 es aproximadamente: 4.937168


4.937168